In [5]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [6]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [7]:
import random

def encode_qubit(bit, basis):
    """Alice encodes a bit in a given basis (0=rectilinear, 1=diagonal)."""
    qc = QuantumCircuit(1)
    if bit == 1:
        qc.x(0)          # Flip to |1⟩
    if basis == 1:
        qc.h(0)          # Rotate to diagonal basis
    return qc

def measure_qubit(qc, basis):
    """Measure a qubit in a given basis. Returns a new circuit with measurement."""
    qc_m = qc.copy()
    if basis == 1:
        qc_m.h(0)        # Rotate back from diagonal before measuring
    qc_m.measure_all()
    return qc_m

def simulate(qc):
    """Run circuit on BasicSimulator and return the measured bit."""
    backend = BasicSimulator()
    compiled = transpile(qc, backend)
    result = backend.run(compiled, shots=1).result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

In [15]:
def bb84_with_eve(n_bits=20):
    # Alice generates random bits and bases
    alice_bits   = [random.randint(0, 1) for _ in range(n_bits)]
    alice_bases  = [random.randint(0, 1) for _ in range(n_bits)]

    # Eve intercepts: random bases, measures & resends
    eve_bases    = [random.randint(0, 1) for _ in range(n_bits)]
    eve_results  = []
    eve_circuits = []   # What Eve resends to Bob

    for i in range(n_bits):
        alice_qc = encode_qubit(alice_bits[i], alice_bases[i])
        meas_qc  = measure_qubit(alice_qc, eve_bases[i])
        eve_bit  = simulate(meas_qc)
        eve_results.append(eve_bit)
        # Eve resends a fresh qubit based on what she measured
        eve_circuits.append(encode_qubit(eve_bit, eve_bases[i]))

    # Bob measures Eve's resent qubits in random bases
    bob_bases   = [random.randint(0, 1) for _ in range(n_bits)]
    bob_results = []

    for i in range(n_bits):
        meas_qc  = measure_qubit(eve_circuits[i], bob_bases[i])
        bob_bit  = simulate(meas_qc)
        bob_results.append(bob_bit)

    # Sifting: keep only bits where Alice & Bob used same basis
    sifted_alice = []
    sifted_bob   = []
    for i in range(n_bits):
        if alice_bases[i] == bob_bases[i]:
            sifted_alice.append(alice_bits[i])
            sifted_bob.append(bob_results[i])

    # Error rate calculation
    errors = sum(a != b for a, b in zip(sifted_alice, sifted_bob))
    error_rate = errors / len(sifted_alice) if sifted_alice else 0

    # Report
    print(f"Total bits sent:      {n_bits}")
    print(f"Bits after sifting:   {len(sifted_alice)}")
    print(f"Errors detected:      {errors}")
    print(f"Error rate:           {error_rate:.1%}")
    print()
    if error_rate > 0.1:
        print("High error rate detected — Eve is present.")
    else:
        print("Low error rate — channel appears secure.")

    return sifted_alice, sifted_bob, error_rate

bb84_with_eve(n_bits=100)

Total bits sent:      100
Bits after sifting:   47
Errors detected:      11
Error rate:           23.4%

High error rate detected — Eve is present.


([0,
  1,
  0,
  1,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  1,
  1,
  0,
  1,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  1,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  1,
  0,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  0,
  1,
  1],
 [0,
  0,
  1,
  1,
  0,
  1,
  1,
  0,
  0,
  1,
  0,
  1,
  1,
  0,
  1,
  0,
  0,
  1,
  0,
  1,
  1,
  0,
  0,
  1,
  0,
  0,
  1,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  1,
  1,
  1,
  0,
  1,
  1],
 0.23404255319148937)